In [1]:
!git clone https://github.com/AshishJangra27/datasets/
!pip install -q chromadb sentence-transformers google-generativeai

Cloning into 'datasets'...
remote: Enumerating objects: 367, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 367 (delta 30), reused 54 (delta 9), pack-reused 235 (from 1)
Receiving objects: 100% (367/367), 287.89 MiB | 17.11 MiB/s, done.
Resolving deltas: 100% (156/156), done.
Updating files: 100% (235/235), done.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.5 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.

In [2]:
import os
import json
import pandas as pd
from tqdm.auto import tqdm

data_path = '/content/datasets/Intervew Questions/'   # note: the folder name is spelled "Intervew" in the repo

csvs = [csv for csv in os.listdir(data_path) if csv.endswith('.csv')]
print("CSV files found:", csvs)

df = pd.DataFrame()
for csv in tqdm(csvs):
    df_ = pd.read_csv(data_path + csv)
    df = pd.concat((df, df_))

df = df.reset_index(drop=True)

# Build the text that will be embedded
df['context'] = "Question : " + df['question'].astype(str) + " asked in the company " + df['company'].astype(str) + " for the position of " + df['position'].astype(str)

print(df.shape)
print(df.columns.tolist())
df.head()

CSV files found: ['ai.csv', 'data_analytics.csv', 'data_science.csv']


  0%|          | 0/3 [00:00<?, ?it/s]

(23865, 7)
['question', 'company', 'rating', 'position', 'date', 'comments', 'context']


,question,company,rating,position,date,comments,context
0,Basic algos questions,Interviewed at Google,4.3,AI Researcher,29 Mar 2019,3,Question : Basic algos questions asked in the ...
1,4 16 64 256 solve using for loop.,Interviewed at Trutech Web Solutions,2.9,Java Developer AI,1 Dec 2015,3,Question : 4 16 64 256 solve using for loop. a...
2,Tell us about your projects and resume,Interviewed at Pirimid Fintech,4.7,AI/ML Engineer,9 Jul 2021,2,Question : Tell us about your projects and res...
3,Assignment was building an algorithm to play t...,Interviewed at NewsBytes,3.3,AI Engineer,6 May 2024,2,Question : Assignment was building an algorith...
4,How to make comments classification?,Interviewed at zarplata.ru,5.0,AI/ML Engineer,27 Oct 2021,1,Question : How to make comments classification...


In [3]:
!pip install -q llama-index

from llama_index.core.node_parser import SentenceSplitter

text_splitter = SentenceSplitter(chunk_size=200, chunk_overlap=50)

chunked_data = []
for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    question = row['question']
    if isinstance(question, str) and question.strip():
        chunks = text_splitter.split_text(question)
        for chunk_no, chunk in enumerate(chunks):
            chunked_data.append({
                'original_id': index,
                'chunk_no': chunk_no,
                'chunk': f"Question : {chunk} asked in the company {row['company']} for the position of {row['position']}",
                'company': row['company'],
                'position': row['position'],
            })

chunked_df = pd.DataFrame(chunked_data)

print("Original rows :", df.shape[0])
print("Total chunks  :", chunked_df.shape[0])
print("Rows that got split into 2+ chunks:", (chunked_df['chunk_no'] > 0).sum())
print()
print(chunked_df.iloc[0]['chunk'])
chunked_df.head()

  0%|          | 0/23865 [00:00<?, ?it/s]

Original rows : 23865
Total chunks  : 24005
Rows that got split into 2+ chunks: 146

Question : Basic algos questions asked in the company Interviewed at Google for the position of AI Researcher


,original_id,chunk_no,chunk,company,position
0,0,0,Question : Basic algos questions asked in the ...,Interviewed at Google,AI Researcher
1,1,0,Question : 4 16 64 256 solve using for loop. a...,Interviewed at Trutech Web Solutions,Java Developer AI
2,2,0,Question : Tell us about your projects and res...,Interviewed at Pirimid Fintech,AI/ML Engineer
3,3,0,Question : Assignment was building an algorith...,Interviewed at NewsBytes,AI Engineer
4,4,0,Question : How to make comments classification...,Interviewed at zarplata.ru,AI/ML Engineer


In [4]:
chroma_data = []

for index, row in tqdm(chunked_df.iterrows(), total=chunked_df.shape[0]):
    chroma_data.append({
        'ids': f"{row['original_id']}_{index}",      # unique ID per chunk
        'documents': row['chunk'],                   # text that will be embedded
        'metadatas': {
            'original_id': int(row['original_id']),
            'chunk_no': int(row['chunk_no']),
            'company': "" if pd.isna(row['company']) else str(row['company']),
            'position': "" if pd.isna(row['position']) else str(row['position']),
        }
    })

print("Total records:", len(chroma_data))
print(json.dumps(chroma_data[0], indent=2))

  0%|          | 0/24005 [00:00<?, ?it/s]

Total records: 24005
{
  "ids": "0_0",
  "documents": "Question : Basic algos questions asked in the company Interviewed at Google for the position of AI Researcher",
  "metadatas": {
    "original_id": 0,
    "chunk_no": 0,
    "company": "Interviewed at Google",
    "position": "AI Researcher"
  }
}


In [5]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from chromadb.config import Settings
import torch

client = chromadb.PersistentClient(path="chroma_db_interview", settings=Settings())

collection_name = "interview_collection"

# Delete the old collection if it exists, so we don't get duplicates
try:
    client.delete_collection(name=collection_name)
    print(f"Deleted old collection '{collection_name}'.")
except Exception:
    print(f"No existing collection named '{collection_name}'.")

# Use GPU if available, otherwise fall back to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

ef = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2", device=device)

collection = client.get_or_create_collection(name=collection_name, embedding_function=ef)
print(f"Created collection '{collection_name}'. Current count: {collection.count()}")

No existing collection named 'interview_collection'.
Using device: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Created collection 'interview_collection'. Current count: 0


In [6]:
batch_size = 256

for i in tqdm(range(0, len(chroma_data), batch_size)):
    batch = chroma_data[i : i + batch_size]
    collection.add(
        ids=[item['ids'] for item in batch],
        documents=[item['documents'] for item in batch],
        metadatas=[item['metadatas'] for item in batch]
    )

print(f"Added {len(chroma_data)} records to '{collection_name}'.")
print(f"Final document count in Chroma: {collection.count()}")

  0%|          | 0/94 [00:00<?, ?it/s]

Added 24005 records to 'interview_collection'.
Final document count in Chroma: 24005


In [7]:
query_text = "Questions asked for Data Scientist position"   # change this to anything you like
n_results = 5

results = collection.query(
    query_texts=[query_text],   # Chroma embeds the query for you
    n_results=n_results
)

if results and results.get('documents') and results['documents'][0]:
    print(f"Query: {query_text}")
    print(f"Top {n_results} matches:")
    for i in range(len(results['documents'][0])):
        document = results['documents'][0][i]
        metadata = results['metadatas'][0][i]
        distance = results['distances'][0][i]

        print(f"\n--- Result {i+1} ---")
        print(f"Company : {metadata.get('company', 'N/A')}")
        print(f"Position: {metadata.get('position', 'N/A')}")
        print(f"Distance: {distance:.4f}   (lower = more similar)")
        print(f"Chunk   : {document}")
else:
    print("No results found.")

Query: Questions asked for Data Scientist position
Top 5 matches:

--- Result 1 ---
Company : Interviewed at Aquient
Position: Data Scientist
Distance: 0.1442   (lower = more similar)
Chunk   : Question : Data science Project based questions asked in the company Interviewed at Aquient for the position of Data Scientist

--- Result 2 ---
Company : Interviewed at Vikas Industries
Position: Data Scientist
Distance: 0.1693   (lower = more similar)
Chunk   : Question : And 4-5 questions related to data Science field. asked in the company Interviewed at Vikas Industries for the position of Data Scientist

--- Result 3 ---
Company : Interviewed at Toptal
Position: Data Scientist
Distance: 0.1879   (lower = more similar)
Chunk   : Question : technical questions related to data science asked in the company Interviewed at Toptal for the position of Data Scientist

--- Result 4 ---
Company : Interviewed at Conversocial
Position: Data Scientist
Distance: 0.1940   (lower = more similar)
Chunk   : Q

In [8]:
query_text = "What kind of questions are asked in Google interviews for AI Researcher?"   # change as you like
n_results = 10

results = collection.query(
    query_texts=[query_text],
    n_results=n_results
)

retrieved_chunks = []
if results and results.get('documents') and results['documents'][0]:
    for i in range(len(results['documents'][0])):
        retrieved_chunks.append({
            'company': results['metadatas'][0][i].get('company', 'N/A'),
            'position': results['metadatas'][0][i].get('position', 'N/A'),
            'chunk': results['documents'][0][i]
        })

if retrieved_chunks:
    context = "\n\n".join([
        f"Company: {c['company']}\nPosition: {c['position']}\nContent: {c['chunk']}"
        for c in retrieved_chunks
    ])

    prompt = f"""You are a helpful AI assistant that helps candidates prepare for job interviews.
Use only the following interview records to answer the question.
If the answer is not available in the provided records, say "I cannot answer this question based on the provided information."

Interview Records:
{context}

Question: {query_text}

Answer:
"""
    print(f"Retrieved {len(retrieved_chunks)} chunks. Prompt created.\n")
    print(prompt[:1500])    # preview of the prompt
else:
    prompt = None
    print("No relevant records retrieved, skipping prompt creation.")

Retrieved 10 chunks. Prompt created.

You are a helpful AI assistant that helps candidates prepare for job interviews.
Use only the following interview records to answer the question.
If the answer is not available in the provided records, say "I cannot answer this question based on the provided information."

Interview Records:
Company: Interviewed at Google
Position: AI Researcher
Content: Question : Basic algos questions asked in the company Interviewed at Google for the position of AI Researcher

Company: Interviewed at Google
Position: Google AI Research Intern
Content: Question : Talk about your research, and asked specific questions regarding to your research work asked in the company Interviewed at Google for the position of Google AI Research Intern

Company: Interviewed at Google
Position: AI Researcher
Content: Question : Algorithms asked in the company Interviewed at Google for the position of AI Researcher

Company: Interviewed at DevRev
Position: Applied AI Engineer
Conte

In [10]:
!pip install -q google-genai

from google import genai
from google.colab import userdata

MODEL_NAME = "gemini-3.5-flash-lite"

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

    # Quick test to confirm the key and model work
    test = gemini_client.models.generate_content(
        model=MODEL_NAME,
        contents="Say hello in one short sentence."
    )
    print("Gemini is ready. Test response:", test.text)

except Exception as e:
    print(f"Error setting up Gemini: {e}")
    gemini_client = None

Gemini is ready. Test response: Hello and welcome!


In [11]:
if prompt and gemini_client:
    try:
        response = gemini_client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )
        rag_answer = response.text
        print("--- RAG Based Answer ---\n")
        print(rag_answer)
    except Exception as e:
        print(f"Error generating content from Gemini: {e}")
        rag_answer = "Could not generate answer."
else:
    rag_answer = "Could not generate answer due to missing prompt or model."
    print(rag_answer)

--- RAG Based Answer ---

Based on the provided interview records, the questions asked in Google interviews for an AI Researcher include basic algorithms questions and algorithms questions in general.


In [12]:
def ask_rag(query_text, n_results=10, show_sources=True):
    # 1. Retrieve similar chunks from Chroma
    results = collection.query(query_texts=[query_text], n_results=n_results)

    if not (results and results.get('documents') and results['documents'][0]):
        print("No relevant records found in ChromaDB.")
        return None

    retrieved = []
    for i in range(len(results['documents'][0])):
        retrieved.append({
            'company': results['metadatas'][0][i].get('company', 'N/A'),
            'position': results['metadatas'][0][i].get('position', 'N/A'),
            'chunk': results['documents'][0][i]
        })

    # 2. Build the prompt
    context = "\n\n".join([
        f"Company: {c['company']}\nPosition: {c['position']}\nContent: {c['chunk']}"
        for c in retrieved
    ])

    prompt = f"""You are a helpful AI assistant that helps candidates prepare for job interviews.
Use only the following interview records to answer the question.
Give a clear, structured answer and mention the companies and positions where relevant.
If the answer is not available in the provided records, say "I cannot answer this question based on the provided information."

Interview Records:
{context}

Question: {query_text}

Answer:
"""

    # 3. Generate the answer with Gemini
    try:
        response = gemini_client.models.generate_content(model=MODEL_NAME, contents=prompt)
        answer = response.text
    except Exception as e:
        answer = f"Error generating answer: {e}"

    # 4. Print the result
    print(f"Question: {query_text}\n")
    print("--- RAG Based Answer ---")
    print(answer)

    if show_sources:
        print("\n--- Sources used ---")
        for c in retrieved:
            print(f"- {c['company']} | {c['position']}")

    return answer

# Try it out
_ = ask_rag("What questions are asked in Google interviews for AI Researcher?", n_results=15)

Question: What questions are asked in Google interviews for AI Researcher?

--- RAG Based Answer ---
Based on the provided interview records, the questions asked in Google interviews for the **AI Researcher** position include:

* Basic algorithms questions
* Algorithms questions

--- Sources used ---
- Interviewed at Google | Google AI Research Intern
- Interviewed at Google | AI Researcher
- Interviewed at Google | AI Researcher
- Interviewed at Bloomberg | AI Research Scientist
- Interviewed at Bloomberg | AI Researcher
- Interviewed at SLB | AI Engineer
- Interviewed at DevRev | Applied AI Engineer
- Interviewed at Unknowns | AI Developer
- Interviewed at Mission Cloud | Generative AI Scientist
- Interviewed at Cambia Health Solutions | Machine Learning Scientist
- Interviewed at Maxcode IT Solutions | AI Engineer
- Interviewed at Bending Spoons | AI Research Engineer
- Interviewed at General Motors (GM) | AI Scientist
- Interviewed at Crossover for Work | AI First Software Engineer

In [13]:
import shutil

chroma_db_path = "./chroma_db_interview"
zip_filename = "chroma_db_interview_archive"

try:
    shutil.make_archive("./" + zip_filename, 'zip', chroma_db_path)
    print(f"Successfully created zip archive: ./{zip_filename}.zip")
    print("Download it from the Colab file browser (folder icon on the left).")
except FileNotFoundError:
    print(f"Error: The directory '{chroma_db_path}' was not found.")
except Exception as e:
    print(f"An error occurred while creating the archive: {e}")

Successfully created zip archive: ./chroma_db_interview_archive.zip
Download it from the Colab file browser (folder icon on the left).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')